# 🎨 محوّل الصور الفني — اليوم الوطني السعودي
# AI Style Transfer — Saudi National Day

---

### 🚀 طريقة الاستخدام:
1. شغّل كل الخلايا بالترتيب (`Runtime → Run all`)
2. انتظر حتى يظهر رابط Gradio
3. ارفع صورتك أو التقطها من الكاميرا
4. اختر الأسلوب الفني وشاهد التحول!

### 🚀 How to use:
1. Run all cells in order (`Runtime → Run all`)
2. Wait for the Gradio link to appear
3. Upload a photo or capture from webcam
4. Pick an art style and watch the magic!

---
> **تأكد من تفعيل GPU:** `Runtime → Change runtime type → T4 GPU`

In [ ]:
# ─── Cell 1: Check GPU ───────────────────────────────────────────────────────
import subprocess, sys

result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print(f'✅ GPU متاح: {result.stdout.strip()}')
else:
    print('⚠️  لم يُعثر على GPU — بعض الأساليب ستعمل بشكل أبطأ على CPU')
    print('   لتفعيل GPU: Runtime → Change runtime type → T4 GPU')

In [ ]:
# ─── Cell 2: Install Dependencies ────────────────────────────────────────────
print('📦 تثبيت المكتبات (قد يأخذ 3-5 دقائق في أول مرة)...')

!pip install -q diffusers==0.27.2 transformers==4.40.0 accelerate==0.29.3
!pip install -q gradio==4.29.0
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118 -q
!pip install -q Pillow opencv-python-headless scipy

print('✅ اكتمل تثبيت المكتبات!')

In [ ]:
# ─── Cell 3: Imports & Device Setup ──────────────────────────────────────────
import torch
import numpy as np
import cv2
from PIL import Image, ImageFilter, ImageEnhance
import gradio as gr
from diffusers import StableDiffusionImg2ImgPipeline
import gc
import warnings
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE  = torch.float16 if DEVICE == 'cuda' else torch.float32

print(f'🖥️  الجهاز المستخدم: {DEVICE.upper()}')
if DEVICE == 'cuda':
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Global pipeline holder — loaded once on first use
SD_PIPE = None

In [ ]:
# ─── Cell 4: Model Loader ─────────────────────────────────────────────────────
def load_sd_pipeline(model_id: str = 'Linaqruf/anything-v3.0'):
    """Load Stable Diffusion img2img pipeline (lazy, cached)."""
    global SD_PIPE
    if SD_PIPE is not None:
        return SD_PIPE

    print(f'⬇️  تحميل نموذج SD ({model_id}) — قد يأخذ دقيقتين...')
    SD_PIPE = StableDiffusionImg2ImgPipeline.from_pretrained(
        model_id,
        torch_dtype=DTYPE,
        safety_checker=None,
        requires_safety_checker=False,
    ).to(DEVICE)

    # Memory optimizations for free Colab
    if DEVICE == 'cuda':
        SD_PIPE.enable_attention_slicing()
        SD_PIPE.enable_xformers_memory_efficient_attention()

    print('✅ النموذج جاهز!')
    return SD_PIPE


# Pre-load the model now so the UI starts instantly
print('⬇️  يتم تحميل نموذج الأنمي الآن...')
load_sd_pipeline()
print('✅ النموذج جاهز! يمكنك الآن تشغيل خلية الـ Gradio.')

In [ ]:
# ─── Cell 5: Style Transfer Functions ────────────────────────────────────────

# ---------- OpenCV-based fast styles (no GPU needed) ----------

def pencil_sketch(pil_img: Image.Image) -> Image.Image:
    img = np.array(pil_img.convert('RGB'))
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    inv  = cv2.bitwise_not(gray)
    blur = cv2.GaussianBlur(inv, (21, 21), 0)
    sketch = cv2.divide(gray, cv2.bitwise_not(blur), scale=256.0)
    return Image.fromarray(sketch).convert('RGB')


def watercolor_style(pil_img: Image.Image) -> Image.Image:
    img = np.array(pil_img.convert('RGB'))
    # Bilateral filter repeated for painterly effect
    out = img.copy()
    for _ in range(3):
        out = cv2.bilateralFilter(out, 9, 75, 75)
    # Stylize
    out = cv2.stylization(out, sigma_s=60, sigma_r=0.45)
    return Image.fromarray(out)


def oil_painting_style(pil_img: Image.Image) -> Image.Image:
    img = np.array(pil_img.convert('RGB'))
    # Detail enhance + edge-preserving filter mimics oil paint
    out = cv2.detailEnhance(img, sigma_s=10, sigma_r=0.15)
    out = cv2.edgePreservingFilter(out, flags=2, sigma_s=50, sigma_r=0.4)
    # Boost saturation
    result = Image.fromarray(out)
    result = ImageEnhance.Color(result).enhance(1.6)
    result = ImageEnhance.Contrast(result).enhance(1.2)
    return result


def cartoon_cv_style(pil_img: Image.Image) -> Image.Image:
    img = np.array(pil_img.convert('RGB'))
    # Edge detection + color quantization
    gray  = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    gray  = cv2.medianBlur(gray, 5)
    edges = cv2.adaptiveThreshold(gray, 255,
                                   cv2.ADAPTIVE_THRESH_MEAN_C,
                                   cv2.THRESH_BINARY, 9, 9)
    color = img.copy()
    for _ in range(6):
        color = cv2.bilateralFilter(color, 9, 250, 250)
    cartoon = cv2.bitwise_and(color, color,
                               mask=edges)
    return Image.fromarray(cartoon)


# ---------- Stable Diffusion img2img styles ----------

STYLE_PROMPTS = {
    'anime': (
        'anime style, masterpiece, best quality, detailed face, beautiful eyes, '
        'vibrant colors, Studio Ghibli, smooth shading',
        'lowres, bad anatomy, bad hands, blurry, ugly, deformed'
    ),
    'national_day': (
        'Saudi Arabia national day celebration, traditional Saudi clothing, '
        'green and white colors, ornate illustration, patriotic, detailed artwork',
        'lowres, blurry, ugly'
    ),
    'digital_art': (
        'digital art, concept art, trending on artstation, vibrant, detailed, '
        'painterly, fantasy, epic lighting',
        'lowres, bad anatomy, blurry, ugly'
    ),
}


def sd_style_transfer(
    pil_img: Image.Image,
    style_key: str,
    strength: float = 0.55,
    steps: int = 25,
) -> Image.Image:
    pipe = load_sd_pipeline()
    prompt, neg_prompt = STYLE_PROMPTS[style_key]

    # Resize to SD-friendly size
    img = pil_img.convert('RGB').resize((512, 512), Image.LANCZOS)

    with torch.autocast(DEVICE):
        result = pipe(
            prompt=prompt,
            negative_prompt=neg_prompt,
            image=img,
            strength=strength,
            num_inference_steps=steps,
            guidance_scale=7.5,
        ).images[0]

    # Resize back to original dimensions
    result = result.resize(pil_img.size, Image.LANCZOS)
    torch.cuda.empty_cache()
    return result


# ---------- Master dispatcher ----------

def transform_image(input_img, style_name: str, strength: float):
    if input_img is None:
        return None, '⚠️  الرجاء رفع صورة أولاً'

    pil_img = Image.fromarray(input_img) if isinstance(input_img, np.ndarray) else input_img

    style_map = {
        '✏️ رسم بالرصاص (Pencil Sketch)':     ('cv', pencil_sketch),
        '🎨 ألوان مائية (Watercolor)':          ('cv', watercolor_style),
        '🖼️ لوحة زيتية (Oil Painting)':        ('cv', oil_painting_style),
        '🎭 كرتون (Cartoon)':                   ('cv', cartoon_cv_style),
        '🌸 أنمي (Anime)':                      ('sd', 'anime'),
        '🇸🇦 اليوم الوطني (National Day)':      ('sd', 'national_day'),
        '💻 فن رقمي (Digital Art)':             ('sd', 'digital_art'),
    }

    if style_name not in style_map:
        return None, f'الأسلوب {style_name} غير معروف'

    kind, fn_or_key = style_map[style_name]
    try:
        if kind == 'cv':
            result = fn_or_key(pil_img)
            msg = f'✅ تم التحويل بأسلوب: {style_name}'
        else:
            if DEVICE == 'cpu':
                return None, '⚠️  هذا الأسلوب يحتاج GPU — الرجاء تفعيل T4 في Colab'
            result = sd_style_transfer(pil_img, fn_or_key, strength)
            msg = f'✅ تم التحويل بأسلوب: {style_name}'
    except Exception as e:
        return None, f'❌ خطأ: {str(e)}'

    return np.array(result), msg


print('✅ دوال التحويل جاهزة!')

In [ ]:
# ─── Cell 6: Gradio UI ────────────────────────────────────────────────────────

STYLES = [
    '✏️ رسم بالرصاص (Pencil Sketch)',
    '🎨 ألوان مائية (Watercolor)',
    '🖼️ لوحة زيتية (Oil Painting)',
    '🎭 كرتون (Cartoon)',
    '🌸 أنمي (Anime)',
    '🇸🇦 اليوم الوطني (National Day)',
    '💻 فن رقمي (Digital Art)',
]

CSS = """
#title  { text-align: center; color: #1a7a2a; font-size: 2rem; font-weight: bold; margin-bottom: 4px; }
#sub    { text-align: center; color: #555; font-size: 1rem; margin-bottom: 16px; }
#badge  { text-align: center; font-size: 0.85rem; color: #888; }
.gr-button-primary { background: linear-gradient(135deg, #1a7a2a, #2ecc40) !important;
                     color: white !important; font-size: 1.1rem !important;
                     border-radius: 12px !important; padding: 12px 32px !important; }
.gr-button-primary:hover { opacity: 0.88 !important; }
"""

with gr.Blocks(css=CSS, title='محوّل الصور الفني') as demo:

    gr.Markdown('# 🎨 محوّل الصور الفني', elem_id='title')
    gr.Markdown('التقط صورتك وشاهدها تتحول إلى لوحة فنية رائعة!', elem_id='sub')
    gr.Markdown('اليوم الوطني السعودي 🇸🇦 | Saudi National Day', elem_id='badge')

    with gr.Row():
        with gr.Column(scale=1):
            input_img = gr.Image(
                label='📷 صورتك — Your Photo',
                sources=['upload', 'webcam', 'clipboard'],
                type='numpy',
                height=350,
            )
            style_radio = gr.Radio(
                choices=STYLES,
                value=STYLES[0],
                label='🎭 اختر الأسلوب الفني — Choose Style',
            )
            strength_slider = gr.Slider(
                minimum=0.30,
                maximum=0.80,
                value=0.55,
                step=0.05,
                label='🎚️ قوة التأثير (للأنمي والفن الرقمي فقط) — Style Strength (SD only)',
            )
            transform_btn = gr.Button('✨ حوّل الصورة!  Transform!', variant='primary')

        with gr.Column(scale=1):
            output_img = gr.Image(
                label='🖼️ النتيجة — Result',
                type='numpy',
                height=350,
            )
            status_box = gr.Textbox(
                label='الحالة — Status',
                interactive=False,
                lines=2,
            )

    # Timing note
    gr.Markdown(
        '> **⏱️ أوقات التحويل التقريبية:**  '
        'رسم / مائي / زيتي / كرتون = **فوري (< 5 ثوانٍ)**  |  '
        'أنمي / يوم وطني / فن رقمي = **15-30 ثانية على GPU**'
    )

    gr.Examples(
        examples=[],
        inputs=[input_img],
        label='أمثلة — Examples (ارفع أي صورة شخصية)',
    )

    transform_btn.click(
        fn=transform_image,
        inputs=[input_img, style_radio, strength_slider],
        outputs=[output_img, status_box],
        show_progress='full',
    )

print('🚀 تشغيل الـ Gradio UI...')
demo.launch(share=True, debug=False)

---
## 📝 نصائح للحصول على أفضل نتيجة

| الأسلوب | أفضل نوع صورة | الوقت |
|---------|---------------|-------|
| رسم بالرصاص | أي صورة وجه واضحة | < 2s |
| ألوان مائية | صور ذات ألوان زاهية | < 3s |
| لوحة زيتية | صور طبيعة أو وجه | < 3s |
| كرتون | وجه أمام خلفية بسيطة | < 3s |
| أنمي 🌸 | وجه واضح، إضاءة جيدة | ~20s (GPU) |
| اليوم الوطني 🇸🇦 | صورة شخصية كاملة | ~25s (GPU) |
| فن رقمي | أي صورة | ~20s (GPU) |

### ⚠️ استكشاف الأخطاء
- **CUDA out of memory**: أعد تشغيل الـ runtime (`Runtime → Restart runtime`) ثم أعد تشغيل الخلايا
- **بطء شديد**: تأكد من تفعيل GPU من `Runtime → Change runtime type → T4 GPU`
- **رابط Gradio لا يعمل**: استخدم الرابط الذي يبدأ بـ `gradio.live`

### 💡 تخصيص إضافي
يمكنك تعديل `STYLE_PROMPTS` في الخلية السابقة لإضافة أساليب جديدة أو تغيير الأوصاف.